# Seeding a code

When we start with the group $\langle ZZI, IZZ \rangle$, we can add the stabilizer $XXXX$ (along with a noiseless qubit) to correct phase-flip errors on the first three qubits. What if, instead, the original generators initially have support on the new qubits? We will try this out using $ZZIXI$ and $IZZIX$. Then we will add stabilizers to try to correct arbitrary single-qubit errors.

In [1]:
from typing import List
import numpy as np
import stim
import networkx as nx
from stimcirq import stim_circuit_to_cirq_circuit, cirq_circuit_to_stim_circuit
import cirq
import openfermion as of
from encoded.code_extension import encoding_unitary_for_new_stabilizer
from encoded.utils import cirq_pauli_string_to_stim, stim_pauli_string_to_cirq
from encoded.diagonalizing_circuit import get_measurement_circuit

In [2]:
def all_single_qubit_errors(n: int) -> List[stim.PauliString]:
    errs = []
    for i in range(n):
        for p in range(1, 4):
            pauli_mask = [0] * i + [p] + [0] * (n - i - 1)
            errs.append(stim.PauliString(pauli_mask))
    return errs

In [3]:
errors = all_single_qubit_errors(6)
for err in errors:
    print(err)

+X_____
+Y_____
+Z_____
+_X____
+_Y____
+_Z____
+__X___
+__Y___
+__Z___
+___X__
+___Y__
+___Z__
+____X_
+____Y_
+____Z_
+_____X
+_____Y
+_____Z


In [4]:
def get_uncorrectable_errors(generators):
    number_true = 0
    number_checked = 0
    uncorrectable_errors = []
    for i, ei in enumerate(errors):
        for j in range(i):
            number_checked += 1
            ej = errors[j]
            e = ei * ej
            commutators = []
            for generator in generators:
                comm = e.commutes(generator)
                commutators.append(comm)
            has_anticommuting_operator = any([not b for b in commutators])
            if has_anticommuting_operator:
                number_true += 1
            else:
                print(f"{ei} * {ej} = {e}, {commutators} {has_anticommuting_operator} ")
                uncorrectable_errors.append(e)
    print(f"{number_true}/{number_checked} operators anticommute.")
    return uncorrectable_errors

In [5]:
generators = [
    stim.PauliString("XXIIZZ"),
    stim.PauliString("IIIZIZ"),
    stim.PauliString("IZIXIX"),
    stim.PauliString("ZZIYYY")
]

In [6]:
for gen1 in generators:
    for gen2 in generators:
        assert gen1.commutes(gen2), f"[{gen1}, {gen2}] != 0"

In [7]:
errors_bad = get_uncorrectable_errors(generators)

+_Z____ * +Z_____ = +ZZ____, [True, True, True, True] False 
+__Y___ * +__X___ = -i__Z___, [True, True, True, True] False 
+__Z___ * +__X___ = +i__Y___, [True, True, True, True] False 
+__Z___ * +__Y___ = -i__X___, [True, True, True, True] False 
+___Z__ * +_X____ = +_X_Z__, [True, True, True, True] False 
+____X_ * +Y_____ = +Y___X_, [True, True, True, True] False 
+____Y_ * +Z_____ = +Z___Y_, [True, True, True, True] False 
+____Y_ * +_Z____ = +_Z__Y_, [True, True, True, True] False 
+____Z_ * +X_____ = +X___Z_, [True, True, True, True] False 
+_____Z * +_X____ = +_X___Z, [True, True, True, True] False 
+_____Z * +___Z__ = +___Z_Z, [True, True, True, True] False 
142/153 operators anticommute.
